In [1]:
import numpy as np
import pandas as pd

In [2]:
train = pd.read_csv('clean-data/sold_train.csv')
val = pd.read_csv('clean-data/sold_validation.csv')
test = pd.read_csv('clean-data/sold_test.csv')

/var/folders/t2/p9112v_n469068__fty8_3nc0000gn/T/ipykernel_93621/2304193386.py:1: DtypeWarning: Columns (39,40) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv('clean-data/sold_train.csv')


## Part 1: Setup
#### 1.1 Load machine learning libraries and preprocessing pipeline

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder
from category_encoders import TargetEncoder
from utilities import impute_groupwise, get_preprocessor

#### 1.2 Select features

In [4]:
train.columns

Index(['Flooring', 'ViewYN', 'PoolPrivateYN', 'ListingKey', 'CloseDate',
       'ClosePrice', 'Latitude', 'Longitude', 'UnparsedAddress', 'LivingArea',
       'AssociationFeeFrequency', 'ListingKeyNumeric', 'MLSAreaMajor',
       'CountyOrParish', 'ElementarySchool', 'AttachedGarageYN',
       'ParkingTotal', 'LotSizeAcres', 'SubdivisionName', 'YearBuilt',
       'StreetNumberNumeric', 'ListingId', 'BathroomsTotalInteger', 'City',
       'BedroomsTotal', 'StateOrProvince', 'MiddleOrJuniorSchool',
       'FireplaceYN', 'Stories', 'HighSchool', 'Levels', 'LotSizeArea',
       'MainLevelBedrooms', 'NewConstructionYN', 'GarageSpaces',
       'HighSchoolDistrict', 'PostalCode', 'AssociationFee',
       'LotSizeSquareFeet', 'OriginatingSystemName',
       'OriginatingSystemSubName', 'latfilled', 'lonfilled'],
      dtype='object')

In [5]:
location_cols = ['MLSAreaMajor', 'CountyOrParish', 'City', 'PostalCode']
bool_cols = ['ViewYN', 'PoolPrivateYN', 'AttachedGarageYN', 'FireplaceYN', 'NewConstructionYN']
numeric_cols = ['LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger', 
                'LotSizeSquareFeet', 'ParkingTotal', 'Stories', 'YearBuilt', 'AssociationFee']
X = location_cols + bool_cols + numeric_cols

#### 1.3 Prepare target and feature sets

In [6]:
X_train = train[X]
y_train = train['ClosePrice']

X_val = val[X]
y_val = val['ClosePrice']

X_test = test[X]
y_test = test['ClosePrice']

#### 1.4 Initiate performance tracker
Create a class to compute, log, and manage evaluation metrics across multiple machine learning models.

In [7]:
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

In [8]:
class ModelPerformanceTracker:
    # set tracker schema
    def __init__(self):
        self.summary = pd.DataFrame(columns=[
            'Model', 'R2', 'RMSE ($)', 'MAE ($)', 'MAPE (%)', 'MdAPE (%)'
        ])

    # calculate evaluation metrics and append them to the summary table
    def log_results(self, model_name, y_true, y_pred):
        r2 = r2_score(y_true, y_pred)
        mae = mean_absolute_error(y_true, y_pred)
        rmse = root_mean_squared_error(y_true, y_pred)

        eps = 1e-8 # have an epsilon in place to prevent division by 0
        percentage_errors = np.abs((y_true - y_pred) / (y_true + eps)) * 100
        mape = np.mean(percentage_errors)
        mdape = np.median(percentage_errors)
        
        model_results = pd.DataFrame([{
            'Model': model_name,
            'R2': round(r2, 4),
            'RMSE ($)': round(rmse, 2),
            'MAE ($)': round(mae, 2), 
            'MAPE (%)': round(mape, 2),
            'MdAPE (%)': round(mdape, 2)
        }])
        self.summary = pd.concat([self.summary, model_results], ignore_index=True)

    # return model performance summary sorted by MdAPE  
    def get_summary(self):
        return self.summary.sort_values(by='MdAPE (%)', ascending=True).reset_index(drop=True)

In [9]:
# initiate the tracker
tracker = ModelPerformanceTracker()

## Part 2: Modeling
#### 2.1 Baseline Models
Build linear regression models with a log-transformed target, beginning with ordinary least squares (OLS) and then extending to ridge regression.

In [10]:
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import LinearRegression, Ridge

In [11]:
linear_preprocessor = get_preprocessor(
    location_cols, 
    bool_cols, 
    numeric_cols,
    scale_skewed=True
)

In [12]:
ols_pipeline = Pipeline([
    ('features', linear_preprocessor),
    ('estimator', LinearRegression())
])

ols_model = TransformedTargetRegressor(
    regressor=ols_pipeline, func=np.log1p, inverse_func=np.expm1
)

ols_model.fit(X_train, y_train)
ols_pred_val = ols_model.predict(X_val)
tracker.log_results(
    model_name='Linear Regression (OLS)', 
    y_true=y_val, 
    y_pred=ols_pred_val
)

/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/

In [13]:
ridge_pipeline = Pipeline([
    ('features', linear_preprocessor),
    ('estimator', Ridge(alpha=1.0)) # L2 penalty
])

ridge_model = TransformedTargetRegressor(
    regressor=ridge_pipeline, func=np.log1p, inverse_func=np.expm1
)

ridge_model.fit(X_train, y_train)
ridge_pred_val = ridge_model.predict(X_val)
tracker.log_results(
    model_name='Linear Regression (L2 Regularization)',
    y_true=y_val,
    y_pred=ridge_pred_val
)

/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/

In [14]:
tracker.get_summary()

,Model,R2,RMSE ($),MAE ($),MAPE (%),MdAPE (%)
0,Linear Regression (OLS),0.7987,418769.22,219631.24,16.56,12.42
1,Linear Regression (L2 Regularization),0.7987,418772.64,219632.27,16.56,12.42


#### 2.2 Tree-Based Models
Build tree-based models with a log-transformed target, beginning with decision tree and then extending to random forest.

In [15]:
from sklearn.compose import TransformedTargetRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

In [16]:
tree_preprocessor = get_preprocessor(
    location_cols, 
    bool_cols, 
    numeric_cols,
    scale_skewed=False # no need to normalize numeric features for tree architectures
)

Decision trees:

In [17]:
# parameter to tune
depth = 8

In [18]:
dt_pipeline = Pipeline([
    ('features', tree_preprocessor),
    ('estimator', DecisionTreeRegressor(max_depth=depth, random_state=42))
])

dt_model = TransformedTargetRegressor(
    regressor=dt_pipeline, func=np.log1p, inverse_func=np.expm1
)

dt_model.fit(X_train, y_train)
dt_pred_val = dt_model.predict(X_val)

# Log results to your existing performance tracker
tracker.log_results(
    model_name=f'Decision Tree (Depth={depth})',
    y_true=y_val,
    y_pred=dt_pred_val
)

/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/

Random forests:

In [19]:
# parameters to tune
n_trees = 150
depth = 12

In [20]:
rf_pipeline = Pipeline([
    ('features', tree_preprocessor),
    ('estimator', RandomForestRegressor(n_estimators=n_trees, max_depth=depth, random_state=42, n_jobs=-1))
])

rf_model = TransformedTargetRegressor(
    regressor=rf_pipeline, func=np.log1p, inverse_func=np.expm1
)

rf_model.fit(X_train, y_train)
rf_pred_val = rf_model.predict(X_val)
tracker.log_results(
    model_name=f'Random Forest ({n_trees} Trees, Depth={depth})',
    y_true=y_val,
    y_pred=rf_pred_val
)

/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/

In [21]:
tracker.get_summary()

,Model,R2,RMSE ($),MAE ($),MAPE (%),MdAPE (%)
0,"Random Forest (150 Trees, Depth=12)",0.8691,337740.94,178138.94,13.06,9.41
1,Decision Tree (Depth=8),0.8314,383279.26,211914.87,15.87,11.83
2,Linear Regression (OLS),0.7987,418769.22,219631.24,16.56,12.42
3,Linear Regression (L2 Regularization),0.7987,418772.64,219632.27,16.56,12.42
